# Cluster-based Summarization


## Import packages


In [11]:
%load_ext autoreload
%autoreload 2
import json
from itertools import product
import numpy as np
import os
from time import time
from tqdm.notebook import tqdm
from argsum import (
    load_test_df,
    get_summetix_cluster_sums,
    get_t5_cluster_sums,
    get_llm_cluster_sums,
)
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data


In [12]:
ArgKP21 = load_test_df("ArgKP21")
Debate_test = load_test_df("Debate_test")

## Define functions


In [13]:
import traceback


def get_cluster_sums(
    df,
    cluster_dict,
    get_cluster_sums_callable,
    parameter_dict,
    output_dir="investigations/2_cluster_summaries",
    file_name=None,
):

    # Get cluster parameter names and values
    clu_parameter_names = cluster_dict["parameter_names"]
    clu_parameter_values = cluster_dict["parameter_values"]
    clu_parameter_combinations = list(product(*clu_parameter_values))

    # Get unique topics and stances
    topics = df["topic"].unique().tolist()
    stances = [str(int(sta)) for sta in sorted(df["stance"].unique())]

    # Get parameter for iteration
    iterate_parameter_names = [
        item[0] for item in parameter_dict.items() if type(item[1]) == list
    ]
    iterate_parameter_values = [
        parameter_dict[parameter_name] for parameter_name in iterate_parameter_names
    ]
    iter_parameter_value_combinations = list(product(*iterate_parameter_values))

    # Create empty dict to store the clusters
    results = dict(
        zip(
            [
                "summaries",
                "clu_parameter_names",
                "clu_parameter_values",
                "sum_parameter_names",
                "sum_parameter_values",
            ],
            [
                dict(
                    zip(
                        [str(comb) for comb in clu_parameter_combinations],
                        [
                            dict(
                                zip(
                                    topics,
                                    [
                                        dict(
                                            zip(
                                                stances,
                                                [
                                                    dict(
                                                        zip(
                                                            [
                                                                str(comb)
                                                                for comb in iter_parameter_value_combinations
                                                            ],
                                                            [
                                                                dict(
                                                                    zip(
                                                                        [
                                                                            "sums",
                                                                            "runtime",
                                                                        ],
                                                                        [None, None],
                                                                    )
                                                                )
                                                                for i in range(
                                                                    len(
                                                                        iter_parameter_value_combinations
                                                                    )
                                                                )
                                                            ],
                                                        )
                                                    )
                                                    for i in range(len(stances))
                                                ],
                                            )
                                        )
                                        for i in range(len(topics))
                                    ],
                                )
                            )
                            for i in range(len(clu_parameter_combinations))
                        ],
                    )
                )
            ]
            + [
                dict(
                    zip(
                        [str(comb) for comb in clu_parameter_combinations],
                        [
                            dict(
                                zip(
                                    topics,
                                    [
                                        dict(
                                            zip(
                                                stances,
                                                [
                                                    dict(
                                                        zip(
                                                            [
                                                                str(comb)
                                                                for comb in iter_parameter_value_combinations
                                                            ],
                                                            [
                                                                dict(
                                                                    zip(
                                                                        [
                                                                            "sums",
                                                                            "runtime",
                                                                        ],
                                                                        [None, None],
                                                                    )
                                                                )
                                                                for i in range(
                                                                    len(
                                                                        iter_parameter_value_combinations
                                                                    )
                                                                )
                                                            ],
                                                        )
                                                    )
                                                    for i in range(len(stances))
                                                ],
                                            )
                                        )
                                        for i in range(len(topics))
                                    ],
                                )
                            )
                            for i in range(len(clu_parameter_combinations))
                        ],
                    )
                )
            ]
            + [
                clu_parameter_names,
                clu_parameter_values,
                iterate_parameter_names,
                iterate_parameter_values,
            ],
        )
    )

    ################################
    ### Iterate: topic & stance ####
    ################################

    for topic_stance in tqdm(
        [(topic, stance) for topic in topics for stance in stances],
        leave=True,
        desc="topic + stance",
    ):

        topic = topic_stance[0]
        stance = topic_stance[1]
        mask_topic_stance = (df["topic"] == topic) & (df["stance"] == int(stance))
        arguments = df[mask_topic_stance]["argument"].to_list()

        ##########################################
        ### Iterate: cluster parameter values ####
        ##########################################

        for clu_parameter in tqdm(
            clu_parameter_combinations, leave=True, desc="clustering parameter"
        ):

            if "iterative_clustering" in cluster_dict.keys():
                cluster_ids = cluster_dict["iterative_clustering"][str(clu_parameter)][
                    topic
                ][stance]["cluster_ids"]
                clustering_runtime = cluster_dict["iterative_clustering"][
                    str(clu_parameter)
                ][topic][stance]["runtime"]
            else:
                cluster_ids = cluster_dict["clustering"][str(clu_parameter)][topic][
                    stance
                ]["cluster_ids"]
                clustering_runtime = cluster_dict["clustering"][str(clu_parameter)][
                    topic
                ][stance]["runtime"]

            cluster_ids_no_noise = [
                cluster_ids[i] for i in range(len(cluster_ids)) if cluster_ids[i] != -1
            ]
            arguments_no_noise = [
                arguments[i] for i in range(len(cluster_ids)) if cluster_ids[i] != -1
            ]

            cond_1 = len(set(cluster_ids_no_noise)) > 1  # Number of clusters > 1
            cond_2 = (
                len(cluster_ids_no_noise) / len(cluster_ids)
            ) > 0.5  # Proportion of clustered arguments > 50%

            ############################
            ### Iterate: parameter #####
            ############################

            # Only if the conditions are true
            if cond_1 & cond_2:

                for comb in tqdm(
                    iter_parameter_value_combinations,
                    leave=False,
                    disable=True,
                    desc="summarization parameter",
                ):
                    iterate_parameter_dict = {
                        **parameter_dict,
                        **dict(zip(iterate_parameter_names, list(comb))),
                    }

                    ########################
                    ### Get summaries ######
                    ########################

                    try:
                        start_time = time()
                        cluster_sums = get_cluster_sums_callable(
                            arguments_no_noise,
                            cluster_ids_no_noise,
                            topic=topic,
                            stance=int(stance),
                            **iterate_parameter_dict,
                        )
                        runtime = time() - start_time
                    except Exception as e:
                        print(f"Error: {e}")
                        print(f"Topic: {topic}, Stance: {stance}")
                        print(f"Cluster Parameter: {clu_parameter}")
                        print(f"Iteration Parameter Combination: {comb}")
                        traceback.print_exc()

                        cluster_sums = None
                        runtime = None

                    results["summaries"][str(clu_parameter)][topic][stance][str(comb)][
                        "sums"
                    ] = cluster_sums
                    if runtime != None:
                        results["summaries"][str(clu_parameter)][topic][stance][
                            str(comb)
                        ]["runtime"] = np.round(clustering_runtime + runtime, 3)

    ########################
    ### Save results #######
    ########################

    if file_name != None:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        with open(output_dir + "/" + file_name, "w") as file:
            json.dump(results, file)

    return results

## Summetix


In [ ]:
with open("investigations/1_argument_clusters/ArgKP21_Summetix.json") as f:
    summetix_cluster_dict = json.load(f)

summetix_parameter_dict = {}

summetix_results = get_cluster_sums(
    df=ArgKP21,
    cluster_dict=summetix_cluster_dict,
    get_cluster_sums_callable=get_summetix_cluster_sums,
    parameter_dict=summetix_parameter_dict,
    file_name="ArgKP21_Summetix.json",
)

In [ ]:
with open("investigations/1_argument_clusters/Debate_test_Summetix.json") as f:
    summetix_cluster_dict = json.load(f)

summetix_parameter_dict = {}

summetix_results = get_cluster_sums(
    df=Debate_test,
    cluster_dict=summetix_cluster_dict,
    get_cluster_sums_callable=get_summetix_cluster_sums,
    parameter_dict=summetix_parameter_dict,
    file_name="Debate_test_Summetix.json",
)

## USKPM


In [ ]:
with open("investigations/1_argument_clusters/ArgKP21_USKPM.json") as f:
    uskpm_cluster_dict = json.load(f)

uskpm_parameter_dict = {
    "max_length_text": 512,
    "max_length_label": 128,
    "n": 5,
    "num_beams": 6,
    "temperature": None,
    "do_sample": False,
    "p": None,
}

uskpm_results = get_cluster_sums(
    df=ArgKP21,
    cluster_dict=uskpm_cluster_dict,
    get_cluster_sums_callable=get_t5_cluster_sums,
    parameter_dict=uskpm_parameter_dict,
    file_name="ArgKP21_USKPM.json",
)

topic + stance:   0%|          | 0/6 [00:00<?, ?it/s]

clustering parameter:   0%|          | 0/48 [00:00<?, ?it/s]

loading file spiece.model from cache at /Users/timaltendorf/.cache/huggingface/hub/models--google--flan-t5-base/snapshots/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/spiece.model
loading file tokenizer.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--google--flan-t5-base/snapshots/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--google--flan-t5-base/snapshots/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/special_tokens_map.json
loading file tokenizer_config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--google--flan-t5-base/snapshots/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/tokenizer_config.json
loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config Bert

Get Flan-T5 Cluster Sums:   0%|          | 0/1 [00:00<?, ?it/s]

Generate config GenerationConfig {
  "_from_model_config": true,
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0,
  "transformers_version": "4.28.1"
}



...

Traceback (most recent call last):
  File "/var/folders/t6/06tpct3n0bd5l2pcs9w42pg40000gn/T/ipykernel_27266/723735843.py", line 224, in get_cluster_sums
    cluster_sums = get_cluster_sums_callable(
  File "/Users/timaltendorf/Documents/summetix/ArgSum/argsum/clustering.py", line 667, in get_t5_cluster_sums
    sum = model.generate(
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/torch/autograd/grad_mode.py", line 27, in decorate_context
    return func(*args, **kwargs)
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/transformers/generation/utils.py", line 1524, in generate
    return self.beam_search(
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/transformers/generation/utils.py", line 2860, in beam_search
    next_tokens = next_tokens % vocab_size
NotImplementedError: The operator 'aten::remainder.Tensor_out' is not current implemented for the MPS device. If you want this op to be added 

Get Flan-T5 Cluster Sums:   0%|          | 0/1 [00:00<?, ?it/s]

Generate config GenerationConfig {
  "_from_model_config": true,
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0,
  "transformers_version": "4.28.1"
}



Error: The operator 'aten::remainder.Tensor_out' is not current implemented for the MPS device. If you want this op to be added in priority during the prototype phase of this feature, please comment on https://github.com/pytorch/pytorch/issues/77764. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.
Topic: Routine child vaccinations should be mandatory, Stance: -1
Cluster Parameter: (2, 2, 0.4)
Iteration Parameter Combination: ()


Traceback (most recent call last):
  File "/var/folders/t6/06tpct3n0bd5l2pcs9w42pg40000gn/T/ipykernel_27266/723735843.py", line 224, in get_cluster_sums
    cluster_sums = get_cluster_sums_callable(
  File "/Users/timaltendorf/Documents/summetix/ArgSum/argsum/clustering.py", line 667, in get_t5_cluster_sums
    sum = model.generate(
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/torch/autograd/grad_mode.py", line 27, in decorate_context
    return func(*args, **kwargs)
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/transformers/generation/utils.py", line 1524, in generate
    return self.beam_search(
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/transformers/generation/utils.py", line 2860, in beam_search
    next_tokens = next_tokens % vocab_size
NotImplementedError: The operator 'aten::remainder.Tensor_out' is not current implemented for the MPS device. If you want this op to be added 

In [ ]:
with open("investigations/1_argument_clusters/Debate_test_USKPM.json") as f:
    uskpm_cluster_dict = json.load(f)

uskpm_parameter_dict = {
    "max_length_text": 512,
    "max_length_label": 128,
    "n": 5,
    "num_beams": 6,
    "temperature": None,
    "do_sample": False,
    "p": None,
}

uskpm_results = get_cluster_sums(
    df=Debate_test,
    cluster_dict=uskpm_cluster_dict,
    get_cluster_sums_callable=get_t5_cluster_sums,
    parameter_dict=uskpm_parameter_dict,
    file_name="Debate_test_USKPM.json",
)

## MCArgSum


### Local


In [ ]:
with open(
    "investigations/1_argument_clusters/ArgKP21_MCArgSum_SBERT_all_mpnet_base.json"
) as f:
    mc_argsum_cluster_dict = json.load(f)

mc_argsum_local_parameter_dict = {
    "llm": "gpt-4o",
    "optimization": "local",
    "sum_token_length": 12,
    "sum_min_num": 1,
    "sum_max_num": 1,
    "few_shot": True,
    "exclude_topic": False,
    "generate_less": False,
    "temperature": 0.5,
    "frequency_penalty": None,
    "n": 5,
    "p": None,
}

mc_argsum_local_results = get_cluster_sums(
    df=ArgKP21,
    cluster_dict=mc_argsum_cluster_dict,
    get_cluster_sums_callable=get_llm_cluster_sums,
    parameter_dict=mc_argsum_local_parameter_dict,
    file_name="ArgKP21_MCArgSum_SBERT_all_mpnet_base_local.json",
)

In [ ]:
with open(
    "investigations/1_argument_clusters/Debate_test_MCArgSum_SBERT_all_mpnet_base.json"
) as f:
    mc_argsum_cluster_dict = json.load(f)

mc_argsum_local_parameter_dict = {
    "llm": "gpt-4o",
    "optimization": "local",
    "sum_token_length": 12,
    "sum_min_num": 1,
    "sum_max_num": 1,
    "few_shot": True,
    "exclude_topic": False,
    "generate_less": False,
    "temperature": 0.5,
    "frequency_penalty": None,
    "n": 5,
    "p": None,
}

mc_argsum_local_results = get_cluster_sums(
    df=Debate_test,
    cluster_dict=mc_argsum_cluster_dict,
    get_cluster_sums_callable=get_llm_cluster_sums,
    parameter_dict=mc_argsum_local_parameter_dict,
    file_name="Debate_test_MCArgSum_SBERT_all_mpnet_base_local.json",
)

### Global


In [ ]:
with open(
    "investigations/1_argument_clusters/ArgKP21_MCArgSum_SBERT_all_mpnet_base.json"
) as f:
    mc_argsum_cluster_dict = json.load(f)

mc_argsum_local_parameter_dict = {
    "llm": "gpt-4o",
    "optimization": "global",
    "sum_token_length": 12,
    "sum_min_num": 1,
    "sum_max_num": 1,
    "few_shot": True,
    "exclude_topic": False,
    "generate_less": False,
    "temperature": 0.5,
    "frequency_penalty": None,
    "n": 5,
    "p": None,
}

mc_argsum_local_results = get_cluster_sums(
    df=ArgKP21,
    cluster_dict=mc_argsum_cluster_dict,
    get_cluster_sums_callable=get_llm_cluster_sums,
    parameter_dict=mc_argsum_local_parameter_dict,
    file_name="ArgKP21_MCArgSum_SBERT_all_mpnet_base_global.json",
)

In [ ]:
with open(
    "investigations/1_argument_clusters/Debate_test_MCArgSum_SBERT_all_mpnet_base.json"
) as f:
    mc_argsum_cluster_dict = json.load(f)

mc_argsum_local_parameter_dict = {
    "llm": "gpt-4o",
    "optimization": "global",
    "sum_token_length": 12,
    "sum_min_num": 1,
    "sum_max_num": 1,
    "few_shot": True,
    "exclude_topic": False,
    "generate_less": False,
    "temperature": 0.5,
    "frequency_penalty": None,
    "n": 5,
    "p": None,
}

mc_argsum_local_results = get_cluster_sums(
    df=Debate_test,
    cluster_dict=mc_argsum_cluster_dict,
    get_cluster_sums_callable=get_llm_cluster_sums,
    parameter_dict=mc_argsum_local_parameter_dict,
    file_name="Debate_test_MCArgSum_SBERT_all_mpnet_base_global.json",
)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

# Load JSON files
file1_path = "/Users/timaltendorf/Documents/summetix/ArgSum/investigations/2_cluster_summaries/ArgKP21_MCArgSum_SBERT_all_mpnet_base_global.json"
file2_path = "/Users/timaltendorf/Documents/summetix/ArgSum/investigations/2_cluster_summaries_3.5/ArgKP21_MCArgSum_SBERT_all_mpnet_base_global.json"


def plot_summaries_comparison(file1_path, file2_path):
    with open(file1_path, "r") as f:
        data1 = json.load(f)
        print(data1.keys())

    with open(file2_path, "r") as f:
        data2 = json.load(f)
        print(data2.keys())

    # Extract summaries count for each parameter
    params = sorted(data1["summaries"].keys(), key=lambda x: float(x.strip("(),")))
    counts1 = []
    counts2 = []

    for param in params:
        count1 = sum(
            len(v["()"].get("sums", {}) or {})  # Handle None case
            for topic in data1["summaries"].get(param, {}).values()
            for v in topic.values()
        )
        count2 = sum(
            len(v["()"].get("sums", {}) or {})  # Handle None case
            for topic in data2["summaries"].get(param, {}).values()
            for v in topic.values()
        )
        counts1.append(count1)
        counts2.append(count2)

    print(counts1)
    print(counts2)

    # Compute differences
    differences = np.array(counts1) - np.array(counts2)

    # Plot the results
    plt.figure(figsize=(10, 5))
    plt.plot(params, counts1, marker="o", label="File 1 Summaries")
    plt.plot(params, counts2, marker="s", label="File 2 Summaries")
    plt.bar(params, differences, alpha=0.5, label="Difference (File 1 - File 2)")
    plt.xticks(rotation=45)
    plt.xlabel("Parameter Value")
    plt.ylabel("Number of Summaries")
    plt.legend()
    plt.title(
        f"Comparison of Summaries Generated for Different Parameter Values\n{os.path.basename(file1_path)}"
    )
    plt.grid()
    plt.show()


def plot_summaries_from_dirs(dir1, dir2):
    """
    Plot summaries from files of the same name that are present in both directories.
    """
    files1 = os.listdir(dir1)
    files2 = os.listdir(dir2)
    common_files = list(set(files1) & set(files2))

    for file in common_files:
        plot_summaries_comparison(dir1 + "/" + file, dir2 + "/" + file)


dir1 = "/Users/timaltendorf/Documents/summetix/ArgSum/investigations/3_classification_summaries"
dir2 = "/Users/timaltendorf/Documents/summetix/ArgSum/investigations/3_classification_summaries_3.5"

plot_summaries_from_dirs(dir1, dir2)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os


def parse_param_key(param):
    """Convert parameter keys from string format to tuple of floats, parsing empty strings as 0.0."""
    return tuple(float(x) if x else 0.0 for x in param.strip("() ").split(","))


def plot_summaries_comparison(file1_path, file2_path):
    with open(file1_path, "r") as f:
        data1 = json.load(f)

    with open(file2_path, "r") as f:
        data2 = json.load(f)

    # Extract summaries count for each parameter
    params = sorted(
        set(data1["summaries"].keys()).union(data2["summaries"].keys()),
        key=parse_param_key,
    )
    counts1 = []
    counts2 = []

    for param in params:
        count1 = (
            sum(
                len(v.get("sums", {}))  # Extract from sum_ids list
                for topic in data1["summaries"].get(param, {}).values()
                for v in topic.values()
            )
            if param in data1["summaries"]
            else 0
        )

        count2 = (
            sum(
                len(v.get("sums", {}))  # Extract from sum_ids list
                for topic in data2["summaries"].get(param, {}).values()
                for v in topic.values()
            )
            if param in data2["summaries"]
            else 0
        )

        counts1.append(count1)
        counts2.append(count2)

    print(counts1)
    print(counts2)

    # Compute differences
    differences = np.array(counts1) - np.array(counts2)

    # Plot the results
    plt.figure(figsize=(10, 5))
    plt.plot(params, counts1, marker="o", label="4o Summaries")
    plt.plot(params, counts2, marker="s", label="3.5 Summaries")
    plt.bar(
        range(len(params)), differences, alpha=0.5, label="Difference (File 1 - File 2)"
    )
    plt.xticks(range(len(params)), params, rotation=45)
    plt.xlabel("Parameter Value")
    plt.ylabel("Number of Summaries")
    plt.legend()
    plt.title(
        f"Comparison of Summaries Generated for Different Parameter Values\n{os.path.basename(file1_path)}"
    )
    plt.grid()
    plt.show()


def plot_summaries_from_dirs(dir1, dir2):
    """
    Plot summaries from files of the same name that are present in both directories.
    """
    files1 = os.listdir(dir1)
    files2 = os.listdir(dir2)
    common_files = list(set(files1) & set(files2))

    for file in common_files:
        plot_summaries_comparison(os.path.join(dir1, file), os.path.join(dir2, file))


# Example usage
dir1 = "/Users/timaltendorf/Documents/summetix/ArgSum/investigations/3_classification_summaries"
dir2 = "/Users/timaltendorf/Documents/summetix/ArgSum/investigations/3_classification_summaries_3.5"

plot_summaries_from_dirs(dir1, dir2)